In [1]:
import os
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [2]:
# ============== CONFIG ==============
IMG_SIZE = 256
PATCH_SIZE = 16
BATCH_SIZE = 4
EPOCHS = 25
LR = 1e-4
NUM_CLASSES = 1

# ViT Encoder config (ViT-Base style)
EMBED_DIM = 768
DEPTH = 12
NUM_HEADS = 12
MLP_RATIO = 4.0
DROP_RATE = 0.1
ATTN_DROP = 0.0

# Decoder: 'naive', 'pup', or 'mla'
DECODER_TYPE = 'pup'

BASE_PATH = "/run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task"
TRAIN_IMG = os.path.join(BASE_PATH, "train", "images")
TRAIN_MASK = os.path.join(BASE_PATH, "train", "masks")
TEST_IMG = os.path.join(BASE_PATH, "test", "images")
TEST_MASK = os.path.join(BASE_PATH, "test", "masks")

PRED_BASE = "/run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/SETR"
PRED_MASK_DIR = os.path.join(PRED_BASE, "masks")
OVERLAY_DIR = os.path.join(PRED_BASE, "overlays")
MODEL_SAVE_DIR = os.path.join(os.path.dirname(PRED_BASE), "models")
MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_DIR, "setr_pup.pth")

os.makedirs(PRED_MASK_DIR, exist_ok=True)
os.makedirs(OVERLAY_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

## 1. Patch Embedding

Splits image (H×W×3) into non-overlapping patches of size P×P. Each patch is flattened and linearly projected to `embed_dim`. Learned positional embeddings are added.

In [3]:
class PatchEmbedding(nn.Module):
    """Split image into patches and embed them."""
    def __init__(self, img_size=256, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_dim = in_chans * patch_size * patch_size
        
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)
        
    def forward(self, x):
        # x: (B, 3, H, W) -> (B, embed_dim, H/P, W/P)
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)  # (B, num_patches, embed_dim)
        x = self.norm(x)
        return x  # (B, L, D)

## 2. Vision Transformer Encoder

Standard transformer encoder: Multi-Head Self-Attention + FFN. No [CLS] token—only patch tokens. Global context is modeled in every layer.

In [4]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., drop=0., attn_drop=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=attn_drop, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(mlp_hidden, dim),
            nn.Dropout(drop)
        )
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        xn = self.norm1(x)
        x = x + self.drop(self.attn(xn, xn, xn)[0])
        xn = self.norm2(x)
        x = x + self.drop(self.mlp(xn))
        return x


class VisionTransformerEncoder(nn.Module):
    """ViT Encoder for SETR - encodes image as sequence of patches."""
    def __init__(self, img_size=256, patch_size=16, embed_dim=768, depth=12, num_heads=12,
                 mlp_ratio=4., drop_rate=0.1, attn_drop_rate=0., in_chans=3):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_chans, embed_dim)
        self.num_patches = self.patch_embed.num_patches
        self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.img_size = img_size
        
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.pos_drop = nn.Dropout(p=drop_rate)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, drop_rate, attn_drop_rate)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        
        features = []
        for blk in self.blocks:
            x = blk(x)
            features.append(x)
        x = self.norm(x)
        return x, features  # (B, L, D), list of (B, L, D)

## 3. Decoders

In [5]:
class SETR_Naive_Decoder(nn.Module):
    """Simple decoder: reshape sequence to 2D, conv upsample, classify."""
    def __init__(self, embed_dim, num_classes, img_size, patch_size):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_patches_h = img_size // patch_size
        self.num_patches_w = img_size // patch_size
        
        self.conv = nn.Sequential(
            nn.Conv2d(embed_dim, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Conv2d(256, num_classes, 1)

    def forward(self, x, target_size):
        # x: (B, L, D)
        B, L, D = x.shape
        x = x.transpose(1, 2).view(B, D, self.num_patches_h, self.num_patches_w)
        x = self.conv(x)
        x = F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)
        return self.classifier(x)


class SETR_PUP_Decoder(nn.Module):
    """Progressive UPsampling (PUP): multiple 2x upsample stages."""
    def __init__(self, embed_dim, num_classes, img_size, patch_size):
        super().__init__()
        self.num_patches_h = img_size // patch_size
        self.num_patches_w = img_size // patch_size
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(embed_dim, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        # Progressive upsampling: 16x16 -> 32x32 -> 64x64 -> 128x128 -> 256x256
        scale = 2
        self.upsample_blocks = nn.ModuleList([])
        in_ch = 256
        for _ in range(int(math.log2(img_size // patch_size))):
            self.upsample_blocks.append(nn.Sequential(
                nn.Conv2d(in_ch, in_ch, 3, padding=1),
                nn.BatchNorm2d(in_ch),
                nn.ReLU(inplace=True),
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ))
        self.classifier = nn.Conv2d(in_ch, num_classes, 1)

    def forward(self, x, target_size):
        B, L, D = x.shape
        x = x.transpose(1, 2).view(B, D, self.num_patches_h, self.num_patches_w)
        x = self.conv1(x)
        for block in self.upsample_blocks:
            x = block(x)
        x = F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)
        return self.classifier(x)


class SETR_MLA_Decoder(nn.Module):
    """Multi-Level Aggregation: aggregate features from multiple encoder layers."""
    def __init__(self, embed_dim, num_classes, img_size, patch_size, mla_channels=256, mla_indices=[3, 6, 9, 12]):
        super().__init__()
        self.num_patches_h = img_size // patch_size
        self.num_patches_w = img_size // patch_size
        self.mla_indices = [i - 1 for i in mla_indices if i <= 12]  # 0-indexed
        
        self.mla_heads = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(embed_dim, mla_channels, 3, padding=1),
                nn.BatchNorm2d(mla_channels),
                nn.ReLU(inplace=True),
            ) for _ in self.mla_indices
        ])
        self.fuse = nn.Sequential(
            nn.Conv2d(mla_channels * len(self.mla_indices), 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Conv2d(256, num_classes, 1)

    def forward(self, x, encoder_features, target_size):
        # x is last layer; encoder_features = list of all layer outputs
        B, L, D = x.shape
        
        mla_feats = []
        for i, idx in enumerate(self.mla_indices):
            feat = encoder_features[idx]  # (B, L, D)
            feat = feat.transpose(1, 2).view(B, D, self.num_patches_h, self.num_patches_w)
            feat = self.mla_heads[i](feat)
            mla_feats.append(feat)
        
        x = torch.cat(mla_feats, dim=1)
        x = self.fuse(x)
        x = F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)
        return self.classifier(x)

## 4. SETR Model

In [6]:
class SETR(nn.Module):
    """
    SETR: SEgmentation TRansformer
    Sequence-to-sequence semantic segmentation with ViT encoder.
    """
    def __init__(self, num_classes=1, img_size=256, patch_size=16, embed_dim=768, depth=12,
                 num_heads=12, decoder_type='pup'):
        super().__init__()
        self.encoder = VisionTransformerEncoder(
            img_size=img_size, patch_size=patch_size, embed_dim=embed_dim,
            depth=depth, num_heads=num_heads, drop_rate=DROP_RATE, attn_drop_rate=ATTN_DROP
        )
        
        if decoder_type == 'naive':
            self.decoder = SETR_Naive_Decoder(embed_dim, num_classes, img_size, patch_size)
        elif decoder_type == 'pup':
            self.decoder = SETR_PUP_Decoder(embed_dim, num_classes, img_size, patch_size)
        elif decoder_type == 'mla':
            self.decoder = SETR_MLA_Decoder(embed_dim, num_classes, img_size, patch_size)
        else:
            raise ValueError(f"decoder_type must be naive, pup, or mla, got {decoder_type}")
        
        self.decoder_type = decoder_type
        self.img_size = img_size

    def forward(self, x):
        target_size = (x.shape[2], x.shape[3])
        x, features = self.encoder(x)
        
        if self.decoder_type == 'mla':
            x = self.decoder(x, features, target_size)
        else:
            x = self.decoder(x, target_size)
        return x

## 5. Dataset & DataLoader

In [7]:
class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))
        self.masks = sorted(os.listdir(mask_dir))
        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, self.masks[idx])

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        mask = Image.open(mask_path).convert("L")
        mask = mask.resize((IMG_SIZE, IMG_SIZE))
        mask = np.array(mask).astype("float32") / 255.0
        mask = (mask > 0.5).astype("float32")
        mask = torch.tensor(mask)

        out_name = self.masks[idx]
        return image, mask, out_name


def collate_fn(batch):
    images = torch.stack([b[0] for b in batch])
    masks = torch.stack([b[1] for b in batch])
    names = [b[2] for b in batch]
    return images, masks, names

## 6. Training Setup

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SETR(num_classes=NUM_CLASSES, img_size=IMG_SIZE, patch_size=PATCH_SIZE,
             embed_dim=EMBED_DIM, depth=DEPTH, num_heads=NUM_HEADS, decoder_type=DECODER_TYPE).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(f"Device: {device}")
print(f"Decoder: {DECODER_TYPE}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

Device: cuda
Decoder: pup
Params: 89,977,601


In [9]:
train_dataset = SegDataset(TRAIN_IMG, TRAIN_MASK)
test_dataset = SegDataset(TEST_IMG, TEST_MASK)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

## 7. Training Loop

In [10]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for images, masks, _ in loader:
        images, masks = images.to(device), masks.unsqueeze(1).to(device)
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, masks, _ in loader:
            images, masks = images.to(device), masks.unsqueeze(1).to(device)
            out = model(images)
            loss = criterion(out, masks)
            total_loss += loss.item()
    return total_loss / len(loader)

In [11]:
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)
    print(f"Epoch [{epoch}/{EPOCHS}] Loss: {train_loss:.4f} Val Loss: {val_loss:.4f}")

Epoch [1/25] Loss: 0.1112 Val Loss: 0.0760
Epoch [2/25] Loss: 0.0489 Val Loss: 0.0707
Epoch [3/25] Loss: 0.0393 Val Loss: 0.0388
Epoch [4/25] Loss: 0.0343 Val Loss: 0.0528
Epoch [5/25] Loss: 0.0302 Val Loss: 0.0389
Epoch [6/25] Loss: 0.0278 Val Loss: 0.0458
Epoch [7/25] Loss: 0.0254 Val Loss: 0.0351
Epoch [8/25] Loss: 0.0225 Val Loss: 0.0257
Epoch [9/25] Loss: 0.0211 Val Loss: 0.0231
Epoch [10/25] Loss: 0.0190 Val Loss: 0.0245
Epoch [11/25] Loss: 0.0186 Val Loss: 0.0225
Epoch [12/25] Loss: 0.0167 Val Loss: 0.0227
Epoch [13/25] Loss: 0.0152 Val Loss: 0.0240
Epoch [14/25] Loss: 0.0138 Val Loss: 0.0242
Epoch [15/25] Loss: 0.0131 Val Loss: 0.0270
Epoch [16/25] Loss: 0.0127 Val Loss: 0.0243
Epoch [17/25] Loss: 0.0119 Val Loss: 0.0226
Epoch [18/25] Loss: 0.0115 Val Loss: 0.0261
Epoch [19/25] Loss: 0.0103 Val Loss: 0.0246
Epoch [20/25] Loss: 0.0104 Val Loss: 0.0242
Epoch [21/25] Loss: 0.0095 Val Loss: 0.0240
Epoch [22/25] Loss: 0.0092 Val Loss: 0.0254
Epoch [23/25] Loss: 0.0088 Val Loss: 0.02

In [12]:
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")

Model saved to /run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/models/setr_pup.pth


## 8. Inference & Visualization

In [17]:
model.eval()
torch.set_grad_enabled(False)

with torch.no_grad():
    for images, masks, names in test_loader:
        images = images.to(device)
        preds = model(images)
        preds = torch.sigmoid(preds).cpu().numpy()
        preds_bin = (preds > 0.5).astype(np.uint8)

        for i, name in enumerate(names):
            pred_mask = preds_bin[i, 0]
            mask_out = Image.fromarray((pred_mask * 255).astype(np.uint8))
            mask_out.save(os.path.join(PRED_MASK_DIR, name))

            image_vis = images[i].cpu().permute(1, 2, 0).numpy()
            overlay = image_vis.copy()
            overlay[pred_mask > 0] = [1, 0, 0]
            combined = 0.7 * image_vis + 0.3 * overlay
            combined = (np.clip(combined, 0, 1) * 255).astype(np.uint8)
            Image.fromarray(combined).save(os.path.join(OVERLAY_DIR, name))

print(f"Saved {len(test_dataset)} masks and overlays (filenames match ground truth)")

Saved 860 masks and overlays (filenames match ground truth)


## 9. Test metrics & comparison report

Run **inference** above first so `masks/` and `overlays/` exist. If you skipped training, run **load checkpoint** then **re-run the inference cell**, then metrics and PDF.


In [14]:
# Load saved .pth before inference if you skipped training (after model + DataLoader cells).
# Optional after a fresh train; use before metrics/report when reloading weights only.
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.eval()


SETR(
  (encoder): VisionTransformerEncoder(
    (patch_embed): PatchEmbedding(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
    (pos_drop): Dropout(p=0.1, inplace=False)
    (blocks): ModuleList(
      (0-11): 12 x TransformerBlock(
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.1, inplace=False)
        )
        (drop): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): Laye

In [15]:
# Run after inference so PRED_MASK_DIR is populated.
def dice_np(y_true, y_pred, smooth=1e-6):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred) + smooth)

def iou_np(y_true, y_pred, smooth=1e-6):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - intersection
    return (intersection + smooth) / (union + smooth)

def precision_recall_np(y_true, y_pred):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    return precision, recall

def accuracy_np(y_true, y_pred):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    return np.sum(y_true == y_pred) / len(y_true)

dice_scores, iou_scores, precision_scores, recall_scores, accuracy_scores = [], [], [], [], []
for i in range(len(test_dataset)):
    _, mask, name = test_dataset[i]
    gt = (mask.numpy() > 0.5).astype(np.uint8)
    pred_path = os.path.join(PRED_MASK_DIR, name)
    pred_img = np.array(Image.open(pred_path))
    pred = (pred_img > 127).astype(np.uint8)
    dice_scores.append(dice_np(gt, pred))
    iou_scores.append(iou_np(gt, pred))
    p, r = precision_recall_np(gt, pred)
    precision_scores.append(p)
    recall_scores.append(r)
    accuracy_scores.append(accuracy_np(gt, pred))

print("===== TEST SET RESULTS =====")
print(f"Mean Dice     : {np.mean(dice_scores):.4f}")
print(f"Mean IoU      : {np.mean(iou_scores):.4f}")
print(f"Mean Precision: {np.mean(precision_scores):.4f}")
print(f"Mean Recall   : {np.mean(recall_scores):.4f}")
print(f"Mean Accuracy : {np.mean(accuracy_scores):.4f}")

===== TEST SET RESULTS =====
Mean Dice     : 0.6822
Mean IoU      : 0.5733
Mean Precision: 0.7369
Mean Recall   : 0.6867
Mean Accuracy : 0.9918


In [18]:
# Ground truth vs predicted comparison PDF (Image, GT, Predicted, Overlay, Comparison)
from matplotlib.backends.backend_pdf import PdfPages

SAMPLES_PER_PAGE = 4
COMPARISON_PDF_PATH = os.path.join(PRED_BASE, "comparison_report.pdf")


def make_comparison_overlay(gt_mask, pred_mask):
    """TP=green, FP=red, FN=blue."""
    gt = (gt_mask > 0.5).astype(np.uint8)
    pred = (pred_mask > 0.5).astype(np.uint8)
    h, w = gt.shape
    overlay = np.zeros((h, w, 3), dtype=np.uint8)
    overlay[gt == 1] = [0, 0, 128]
    overlay[pred == 1] = [128, 0, 0]
    overlay[(gt == 1) & (pred == 1)] = [0, 128, 0]
    return overlay


indices = np.arange(len(test_dataset))

with PdfPages(COMPARISON_PDF_PATH) as pdf:
    for page_start in range(0, len(indices), SAMPLES_PER_PAGE):
        page_indices = indices[page_start : page_start + SAMPLES_PER_PAGE]
        n_rows = len(page_indices)
        fig, axes = plt.subplots(n_rows, 5, figsize=(15, 3 * n_rows))
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        for row, idx in enumerate(page_indices):
            img_name = test_dataset.images[idx]
            mask_name = test_dataset.masks[idx]
            img_path = os.path.join(TEST_IMG, img_name)
            gt_path = os.path.join(TEST_MASK, mask_name)
            pred_path = os.path.join(PRED_MASK_DIR, mask_name)
            overlay_path = os.path.join(OVERLAY_DIR, mask_name)

            image = np.array(Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))) / 255.0
            gt_mask = np.array(Image.open(gt_path).convert("L").resize((IMG_SIZE, IMG_SIZE))) / 255.0
            pred_mask = np.array(Image.open(pred_path).convert("L")) / 255.0
            if pred_mask.shape != (IMG_SIZE, IMG_SIZE):
                pred_mask = np.array(Image.fromarray((pred_mask * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))) / 255.0

            overlay_img = np.array(Image.open(overlay_path).convert("RGB")) / 255.0
            if overlay_img.shape[:2] != (IMG_SIZE, IMG_SIZE):
                overlay_img = np.array(Image.fromarray((overlay_img * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))) / 255.0

            comparison = make_comparison_overlay(gt_mask, pred_mask)

            axes[row, 0].imshow(image)
            axes[row, 0].set_title("Image")
            axes[row, 0].axis("off")

            axes[row, 1].imshow(gt_mask, cmap="gray")
            axes[row, 1].set_title("Ground Truth")
            axes[row, 1].axis("off")

            axes[row, 2].imshow(pred_mask, cmap="gray")
            axes[row, 2].set_title("Predicted")
            axes[row, 2].axis("off")

            axes[row, 3].imshow(overlay_img)
            axes[row, 3].set_title("Overlay")
            axes[row, 3].axis("off")

            axes[row, 4].imshow(comparison)
            axes[row, 4].set_title("Comparison (TP=green, FP=red, FN=blue)")
            axes[row, 4].axis("off")

        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"Comparison PDF saved to {COMPARISON_PDF_PATH}")

Comparison PDF saved to /run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/SETR/comparison_report.pdf
